# Market Sentiment from Financial Tweets: Final Model

Ready-to-run pipeline for market sentiment classification (Bearish / Bullish / Neutral).

**Pipeline:**
1. Load `test.csv` and `train.csv`
2. Preprocess tweets with `clean_tweet_transformer` (regex normalisation only, no stemming or stop-word removal, preserving natural language for the encoder)
3. Encode with **FinTwitBERT** (`StephanAkkerman/FinTwitBERT-sentiment`) CLS embeddings
4. Train an **XGBoost** classifier with pre-tuned hyperparameters
5. Evaluate on the held-out validation set
6. Predict on the test set and save `predictions.csv`

## 0. Imports & Setup

In [9]:
import warnings, re
warnings.filterwarnings('ignore')

import transformers
transformers.logging.set_verbosity_error()

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score,
    f1_score, precision_score, recall_score
)

LABEL_NAMES = ['Bearish (0)', 'Bullish (1)', 'Neutral (2)']
FINTWITBERT_PATH = 'StephanAkkerman/FinTwitBERT-sentiment'

## 1. Load Data

In [10]:
train_df = pd.read_csv('../data/train.csv')
test_df  = pd.read_csv('../data/test.csv')

print(f'Train: {len(train_df)} rows | Test: {len(test_df)} rows')
print(train_df['label'].value_counts().sort_index())

Train: 9543 rows | Test: 2388 rows
label
0    1442
1    1923
2    6178
Name: count, dtype: int64


## 2. Train / Validation Split

In [11]:
train, val = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)
y_val = val['label'].values

print(f'Train: {len(train)} | Val: {len(val)}')

Train: 7634 | Val: 1909


## 3. Preprocessing

`clean_tweet_transformer` applies regex normalisation only: stock tags, ASCII cleanup, URLs, dates, mentions, times and ticker symbols are replaced with semantic placeholder tokens (`>stock`, `>link`, `>date`, `>mention`, `>time`).

Stop-word removal and stemming are intentionally omitted here because FinTwitBERT was pretrained on natural language and performs best on fluent text. Those techniques are only applied for the BoW / Word2Vec features in `01_experiments.ipynb`.

In [12]:
def _date_url_regex(text):
    """Normalise stock tags, URLs, dates, mentions, times and $ tickers."""
    text = re.sub(r'\([A-Z]{1,4}\)', '>stock', text)
    text = text.encode('ascii', 'ignore').decode()
    text = re.sub(r'http\S+|www\S+|https\S+', '>link', text, flags=re.MULTILINE)

    months = (r'(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may'
                r'|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:tember|t)?'
                r'|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)')
    weekdays = (r'(?:mon(?:day)?|tue(?:sday)?|wed(?:nesday)?|thu(?:rsday)?'
                r'|fri(?:day)?|sat(?:urday)?|sun(?:day)?)')

    complex_date = (r'\b(?:' + months +
                    r'\s+\d{1,2}(?:st|nd|rd|th)?(?:,?\s+\d{2,4})?|\d{1,2}(?:st|nd|rd|th)?\s+'
                    + months + r'(?:,?\s+\d{2,4})?)\b')
    text = re.sub(complex_date, '>date', text)
    text = re.sub(r'\b\d+(?:st|nd|rd|th)?\s*(?:days?|weeks?|years?)\b', '>date', text)
    text = re.sub(r'\b(?:' + weekdays + r'|' + months + r')\b', '>date', text)
    text = re.sub(r'@\w+', '>mention', text)
    text = re.sub(r'\b\d{1,2}(?::\d{2})?\s*(?:am|pm)(?:\s+[a-z]{3,4})?\b', '>time', text)
    text = re.sub(r'\$[a-zA-Z]+', '>stock', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def clean_tweet_transformer(text):
    """Regex normalisation only to preserve natural language structure for transformer encoders."""
    return _date_url_regex(str(text))


val_texts  = val['text'].apply(clean_tweet_transformer).tolist()
test_texts = test_df['text'].apply(clean_tweet_transformer).tolist()

for raw, clean in zip(val['text'].head(3), val_texts[:3]):
    print(f'  RAW  : {raw}')
    print(f'  CLEAN: {clean}')
    print()

  RAW  : i love it when fertility clinic's decorate ...... and it's not even Thanks Giving ..... i.e. premature https://t.co/eS8TVNbfJZ
  CLEAN: i love it when fertility clinic's decorate ...... and it's not even Thanks Giving ..... i.e. premature >link

  RAW  : IRS investigators scrutinize Bitcoin ATMs and kiosks
  CLEAN: IRS investigators scrutinize Bitcoin ATMs and kiosks

  RAW  : $SRPT higher by 4.5% today clearing 21 MA and nearing Feb OpEx value range breakout
  CLEAN: >stock higher by 4.5% today clearing 21 MA and nearing Feb OpEx value range breakout



## 4. FinTwitBERT CLS Embeddings

**FinTwitBERT** (`StephanAkkerman/FinTwitBERT-sentiment`) is a BERT encoder fine-tuned on financial tweets.
We extract the `[CLS]` token vector (768-dim) for each tweet as the feature representation fed to the classifier.

In [13]:
print(f'Loading {FINTWITBERT_PATH} ...')
ftb_tokenizer = AutoTokenizer.from_pretrained(FINTWITBERT_PATH)
ftb_model = AutoModel.from_pretrained(FINTWITBERT_PATH)
ftb_model.eval()
print('Model loaded.')


def encode_cls_batched( texts, tokenizer, model, batch_size: int = 32, max_len: int = 128, desc: str = 'Encoding'):
    """Encode texts into CLS token embeddings. Returns (n_texts, hidden_size) array."""
    all_cls = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i: i + batch_size]
        enc   = tokenizer(
            batch, truncation=True, padding=True,
            max_length=max_len, return_tensors='pt'
        )
        with torch.no_grad():
            out = model(**enc)
        all_cls.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(all_cls)


print('Encoding validation set')
X_val  = encode_cls_batched(val_texts,  ftb_tokenizer, ftb_model, desc='FinTwitBERT – val')

print('Encoding test set')
X_test = encode_cls_batched(test_texts, ftb_tokenizer, ftb_model, desc='FinTwitBERT – test')

print(f'Shapes val: {X_val.shape} | test: {X_test.shape}')

Loading StephanAkkerman/FinTwitBERT-sentiment ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded.
Encoding validation set


Encoding test set


Shapes val: (1909, 768) | test: (2388, 768)


## 5. Train Classifier – XGBoost (Tuned)

XGBoost on FinTwitBERT CLS embeddings was one of the two best models in our experiments (macro-F1 0.867, next to the LSTM's 0.869).
The hyperparameters below are the result of a `RandomizedSearchCV` run in the experiments notebook.

In [14]:
from xgboost import XGBClassifier

best_clf = XGBClassifier(
    subsample        = 0.8,
    n_estimators     = 200,
    min_child_weight = 1,
    max_depth        = 8,
    learning_rate    = 0.05,
    gamma            = 0.5,
    colsample_bytree = 1.0,
    eval_metric      = 'mlogloss',
    random_state     = 42,
    n_jobs           = -1
)

# Encode the full training set for fitting
train_texts = train['text'].apply(clean_tweet_transformer).tolist()
print('Encoding training set')
X_train = encode_cls_batched(train_texts, ftb_tokenizer, ftb_model, desc='FinTwitBERT – train')
y_train = train['label'].values

print('\nFitting XGBoost')
best_clf.fit(X_train, y_train)
print('Done.')

Encoding training set



Fitting XGBoost
Done.


## 6. Evaluation on the Validation Set

In [15]:
y_pred_val = best_clf.predict(X_val)

print('=' * 55)
print('  XGBoost (tuned)  |  FinTwitBERT CLS  –  Validation')
print('=' * 55)
print(classification_report(y_val, y_pred_val, target_names=LABEL_NAMES, zero_division=0))
print(f'  Accuracy          : {accuracy_score(y_val, y_pred_val):.4f}')
print(f'  Precision (W)     : {precision_score(y_val, y_pred_val, average="weighted", zero_division=0):.4f}')
print(f'  Recall (W)        : {recall_score(y_val, y_pred_val, average="weighted", zero_division=0):.4f}')
print(f'  F1 Weighted       : {f1_score(y_val, y_pred_val, average="weighted", zero_division=0):.4f}')
print(f'  F1 Macro          : {f1_score(y_val, y_pred_val, average="macro", zero_division=0):.4f}')

  XGBoost (tuned)  |  FinTwitBERT CLS  –  Validation
              precision    recall  f1-score   support

 Bearish (0)       0.84      0.82      0.83       288
 Bullish (1)       0.88      0.79      0.83       385
 Neutral (2)       0.91      0.94      0.93      1236

    accuracy                           0.89      1909
   macro avg       0.88      0.85      0.86      1909
weighted avg       0.89      0.89      0.89      1909

  Accuracy          : 0.8942
  Precision (W)     : 0.8934
  Recall (W)        : 0.8942
  F1 Weighted       : 0.8931
  F1 Macro          : 0.8630


## 7. Generate Test Predictions

In [16]:
final_preds = best_clf.predict(X_test)

pred_df = pd.DataFrame({'id': test_df['id'], 'label': final_preds})
pred_df.to_csv('../predictions.csv', index=False)

print('Saved to predictions.csv')
print('\nPrediction distribution:')
print(pred_df['label'].value_counts().sort_index())
print('\nSample predictions:')
print(pred_df.head(10))

Saved to pred_32.csv

Prediction distribution:
label
0     336
1     431
2    1621
Name: count, dtype: int64

Sample predictions:
   id  label
0   0      1
1   1      2
2   2      2
3   3      2
4   4      2
5   5      2
6   6      2
7   7      2
8   8      2
9   9      2
